In [ ]:
# viz_utils.py (or put into graph_utils.py)
from __future__ import annotations

from typing import Optional, Dict, Tuple
import numpy as np
import matplotlib.pyplot as plt


def _collect_node_arrays(g):
    xs = np.array([n.x for n in g.nodes], dtype=float)
    ys = np.array([n.y for n in g.nodes], dtype=float)
    tags = [n.tag for n in g.nodes]
    return xs, ys, tags


def _default_tag_style() -> Dict[str, Dict]:
    """
    你没要求固定配色，我这里给一个最基本的默认样式。
    需要你自己也可以传入 tag_style 覆盖。
    """
    return {
        "indoor": {"marker": "o", "s": 18, "alpha": 0.9},
        "outdoor": {"marker": "o", "s": 18, "alpha": 0.9},
        "door": {"marker": "s", "s": 30, "alpha": 1.0},
        "no-traversable": {"marker": "x", "s": 30, "alpha": 1.0},
        "unknown": {"marker": ".", "s": 14, "alpha": 0.8},
    }


def _plot_nodes(ax, g, tag_style: Optional[Dict[str, Dict]] = None, show_legend: bool = True):
    xs, ys, tags = _collect_node_arrays(g)
    style = _default_tag_style()
    if tag_style is not None:
        # shallow merge
        for k, v in tag_style.items():
            style.setdefault(k, {}).update(v)

    # 按 tag 分组画（便于 legend）
    uniq_tags = sorted(set(tags))
    for t in uniq_tags:
        idx = [i for i, tt in enumerate(tags) if tt == t]
        st = style.get(t, style["unknown"])
        ax.scatter(xs[idx], ys[idx], label=t, **st)

    if show_legend:
        ax.legend(loc="upper right", frameon=True, fontsize=9)


def _plot_edges(
    ax,
    g,
    *,
    alpha: float = 0.25,
    linewidth: float = 1.0,
    portal_alpha: float = 0.6,
    portal_linewidth: float = 1.5,
    show_portal: bool = True,
):
    """
    画边。默认把 portal 边画得更醒目一点（如果存在）。
    不强制配色（用 matplotlib 默认），但通过 alpha/线宽区分。
    """
    # 为了快：直接用 edges 列表
    for e in g.edges:
        u = g.nodes[e.u]
        v = g.nodes[e.v]

        if show_portal and getattr(e, "etype", None) == "portal":
            ax.plot([u.x, v.x], [u.y, v.y], alpha=portal_alpha, linewidth=portal_linewidth)
        else:
            ax.plot([u.x, v.x], [u.y, v.y], alpha=alpha, linewidth=linewidth)


def plot_graph_on_blank(
    g,
    *,
    width: float,
    height: float,
    title: str = "Graph (blank map)",
    figsize: Tuple[float, float] = (6.5, 6.5),
    show_edges: bool = True,
    show_nodes: bool = True,
    show_legend: bool = True,
    tag_style: Optional[Dict[str, Dict]] = None,
    edge_alpha: float = 0.25,
    edge_linewidth: float = 1.0,
    portal_alpha: float = 0.6,
    portal_linewidth: float = 1.5,
):
    """
    在空白坐标系里绘制 graph（适合 debug 拓扑连接是否合理）
    """
    fig, ax = plt.subplots(figsize=figsize)

    # 空白底图：仅设置范围与网格
    ax.set_xlim(0, width)
    ax.set_ylim(0, height)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.4)

    if show_edges:
        _plot_edges(
            ax, g,
            alpha=edge_alpha, linewidth=edge_linewidth,
            portal_alpha=portal_alpha, portal_linewidth=portal_linewidth,
        )
    if show_nodes:
        _plot_nodes(ax, g, tag_style=tag_style, show_legend=show_legend)

    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    plt.tight_layout()
    plt.show()


def plot_graph_on_costmap(
    costmap: np.ndarray,
    g,
    *,
    title: str = "Graph on costmap",
    figsize: Tuple[float, float] = (6.5, 6.5),
    origin: str = "lower",
    show_colorbar: bool = True,
    show_edges: bool = True,
    show_nodes: bool = True,
    show_legend: bool = True,
    tag_style: Optional[Dict[str, Dict]] = None,
    edge_alpha: float = 0.25,
    edge_linewidth: float = 1.0,
    portal_alpha: float = 0.6,
    portal_linewidth: float = 1.5,
    inf_as_max: bool = True,
):
    """
    在 costmap 上叠加绘制 graph。
    costmap shape = (H, W)，默认 origin="lower" 以匹配你之前的坐标约定。
    """
    T = np.array(costmap, dtype=float)

    if inf_as_max:
        # 为了能画出来，把 inf 替换成 (max_finite * 1.2)
        finite = T[np.isfinite(T)]
        maxv = finite.max() if finite.size > 0 else 1.0
        T = T.copy()
        T[~np.isfinite(T)] = maxv * 1.2

    fig, ax = plt.subplots(figsize=figsize)

    im = ax.imshow(T, origin=origin)
    if show_colorbar:
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    H, W = T.shape
    ax.set_xlim(0, W)
    ax.set_ylim(0, H)
    ax.set_aspect("equal", adjustable="box")

    if show_edges:
        _plot_edges(
            ax, g,
            alpha=edge_alpha, linewidth=edge_linewidth,
            portal_alpha=portal_alpha, portal_linewidth=portal_linewidth,
        )
    if show_nodes:
        _plot_nodes(ax, g, tag_style=tag_style, show_legend=show_legend)

    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    plt.tight_layout()
    plt.show()
